Freddie_Mac_Production_Model.ipynb

# %% [markdown]
# # Freddie Mac IFRS 9 Stress-Testing Engine
# **Author:** Quantitative Credit Modelling  
# **Model Version:** v1.0 (Production)  
# **Dataset:** Freddie Mac Single-Family Loan-Level (Sample) 2006–2023  
# 
# ## 1. Setup & Configuration
# This notebook executes the full end-to-end pipeline: Data Audit, Visual EDA, Transition Matrices, Model Training, and the IFRS 9 ECL Engine. All plots are automatically saved to the `reports/figures` directory for the validation report.

# %%

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, gc, warnings
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Paths
RAW_PATH = r"C:\CreditRiskCourse\data\raw\freddie_mae"
PROCESSED_PATH = r"C:\CreditRiskCourse\data\processed"
MODEL_PATH = r"C:\CreditRiskCourse\models\freddie"
FIGURES_PATH = r"C:\CreditRiskCourse\reports\figures"

for path in [PROCESSED_PATH, MODEL_PATH, FIGURES_PATH]:
    os.makedirs(path, exist_ok=True)

# Macro Data Dictionary
MACRO_DATA = {
    2006: (2.6, 4.6, 7.5), 2007: (1.9, 4.6, 1.5), 2008: (-0.1, 5.8, -9.0), 2009: (-2.5, 9.3, -12.5),
    2010: (2.6, 9.6, -4.0), 2011: (1.6, 8.9, -1.5), 2012: (2.2, 8.1, 4.0), 2013: (1.8, 7.4, 7.0),
    2014: (2.5, 6.2, 5.5), 2015: (2.9, 5.3, 5.0), 2016: (1.6, 4.9, 5.5), 2017: (2.4, 4.4, 6.5),
    2018: (3.0, 3.9, 6.0), 2019: (2.2, 3.7, 4.0), 2020: (-3.4, 8.1, 6.0), 2021: (5.7, 5.3, 15.0),
    2022: (2.1, 3.6, 8.0), 2023: (2.5, 3.6, 1.0)
}

FEATURES = ['CLASSIC FICO', 'ORIGINAL DEBT-TO-INCOME (DTI) RATIO', 'ORIGINAL LOAN-TO-VALUE (LTV)', 
            'ORIGINAL UPB', 'ORIGINAL INTEREST RATE', 'LOAN AGE', 'CURRENT ACTUAL UPB', 
            'CHANNEL', 'PROPERTY TYPE', 'LOAN PURPOSE', 'OCCUPANCY STATUS', 
            'GDP_Growth', 'Unemployment', 'HPI_YoY']
CAT_FEATURES = ['CHANNEL', 'PROPERTY TYPE', 'LOAN PURPOSE', 'OCCUPANCY STATUS']
BASE_COLS = [f for f in FEATURES if f not in ['GDP_Growth', 'Unemployment', 'HPI_YoY']]
NEEDED_COLS = BASE_COLS + ['default_flag']

print("✅ Setup Complete. Directories ready.")


✅ Setup Complete. Directories ready.


# %% [markdown]
# ## 2. Visual EDA & Transition Matrices
# Before trusting the IFRS 9 engine, we must validate the delinquency pipeline. We generate Transition Matrices to prove the data behaves according to credit theory.

# %%

In [3]:
def generate_eda_and_transitions(year_list):
    print("Generating Transition Matrices...")
    for year in year_list:
        file_path = os.path.join(PROCESSED_PATH, f"freddie_master_{year}.parquet")
        if not os.path.exists(file_path): continue
        
        df = pd.read_parquet(file_path, columns=['LOAN IDENTIFIER', 'PERIOD', 'CURRENT LOAN DELINQUENCY STATUS'])
        
        def map_state(s):
            s = str(s).strip()
            if s == '00': return 'Current'
            if s == '01': return '30 DPD'
            if s == '02': return '60 DPD'
            if s == 'RA': return 'REO'
            return '90+ DPD' if s not in ['Unknown'] else 'Unknown'
            
        df['State_t'] = df['CURRENT LOAN DELINQUENCY STATUS'].apply(map_state)
        df = df.sort_values(['LOAN IDENTIFIER', 'PERIOD'])
        df['State_t+1'] = df.groupby('LOAN IDENTIFIER')['State_t'].shift(-1)
        df = df.dropna(subset=['State_t+1'])
        
        counts = pd.crosstab(df['State_t'], df['State_t+1'], normalize='index') * 100
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(counts, annot=True, cmap='Blues', fmt='.2f')
        plt.title(f'Transition Matrix: {year} Vintage')
        plt.ylabel('State at t')
        plt.xlabel('State at t+1')
        
        filename = os.path.join(FIGURES_PATH, f"transition_matrix_{year}.png")
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ Saved: {filename}")

generate_eda_and_transitions([2008, 2016, 2020])


Generating Transition Matrices...
✅ Saved: C:\CreditRiskCourse\reports\figures\transition_matrix_2008.png
✅ Saved: C:\CreditRiskCourse\reports\figures\transition_matrix_2016.png
✅ Saved: C:\CreditRiskCourse\reports\figures\transition_matrix_2020.png


# %% [markdown]
# ## 3. Empirical Lifetime PD Curves
# This chart visually proves the macro-sensitivity of the dataset. The 2006/2008 curves show the slow GFC bleed, while the 2016 curve shows the sudden COVID spike at month 40.

# %%

In [4]:
def plot_pd_curves():
    print("Generating Lifetime PD Curves...")
    vintages = [2006, 2008, 2012, 2016, 2020, 2023]
    results = {}
    
    for year in vintages:
        file_path = os.path.join(PROCESSED_PATH, f"freddie_master_{year}.parquet")
        if not os.path.exists(file_path): continue
        
        df = pd.read_parquet(file_path, columns=['LOAN IDENTIFIER', 'LOAN AGE', 'default_flag'])
        total_loans = df['LOAN IDENTIFIER'].nunique()
        
        df_defaults = df[df['default_flag'] == 1]
        first_default = df_defaults.groupby('LOAN IDENTIFIER')['LOAN AGE'].min().reset_index()
        
        defaults_by_age = first_default['LOAN AGE'].value_counts().sort_index().reset_index()
        defaults_by_age.columns = ['Loan Age', 'Marginal_Defaults']
        
        full_ages = pd.DataFrame({'Loan Age': range(1, 61)})
        defaults_by_age = full_ages.merge(defaults_by_age, on='Loan Age', how='left').fillna(0)
        defaults_by_age['Cumulative_PD_%'] = (defaults_by_age['Marginal_Defaults'].cumsum() / total_loans) * 100
        
        results[year] = defaults_by_age
        del df, df_defaults, first_default
        gc.collect()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    colors = {2006: '#8B0000', 2008: '#FF0000', 2012: '#FFA500', 2016: '#0000FF', 2020: '#800080', 2023: '#008000'}

    for year, df in results.items():
        ax1.plot(df['Loan Age'], df['Cumulative_PD_%'], label=f'{year} Vintage', color=colors[year], linewidth=2.5)
        ax2.plot(df['Loan Age'], df['Marginal_Defaults'], label=f'{year} Vintage', color=colors[year], linewidth=1.5, alpha=0.7)

    ax1.set_title('Cumulative Lifetime PD by Loan Age', fontsize=16, fontweight='bold')
    ax1.set_ylabel('Cumulative Default Rate (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title('Marginal Default Volume (Actual Number of New Defaults per Month)')
    ax2.set_xlabel('Loan Age (Months)')
    ax2.set_ylabel('Number of New Defaults')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = os.path.join(FIGURES_PATH, "lifetime_pd_curves.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {filename}")

plot_pd_curves()


Generating Lifetime PD Curves...
✅ Saved: C:\CreditRiskCourse\reports\figures\lifetime_pd_curves.png


# %% [markdown]
# ## 4. Champion/Challenger Training & Backtesting
# We train an XGBoost Champion and Logistic Regression Challenger using a 3-tier temporal split. 
# **Train:** 2006-2014 | **Validation:** 2015-2016 | **Out-of-Time Backtest:** 2017-2023.

# %%

In [6]:
def train_models():
    print("Training Champion and Challenger models (Memory Safe)...")
    train_list, val_list, oot_list = [], [], []
    
    # Loop through years, sample, and append
    for year in range(2006, 2024):
        file_path = os.path.join(PROCESSED_PATH, f"freddie_master_{year}.parquet")
        if not os.path.exists(file_path): continue
        
        df = pd.read_parquet(file_path, columns=NEEDED_COLS)
        
        # Add Macro
        df['GDP_Growth'] = MACRO_DATA[year][0]
        df['Unemployment'] = MACRO_DATA[year][1]
        df['HPI_YoY'] = MACRO_DATA[year][2]
        
        # Downcast to save RAM
        for col in df.select_dtypes(include=[np.float64]).columns: df[col] = df[col].astype(np.float32)
        
        if year <= 2014:
            train_list.append(df.sample(frac=0.15, random_state=42))
        elif year <= 2016:
            val_list.append(df.sample(frac=0.30, random_state=42))
        else:
            oot_list.append(df.sample(frac=0.30, random_state=42))
            
        del df; gc.collect()

    X_train = pd.concat(train_list, ignore_index=True)
    X_val   = pd.concat(val_list, ignore_index=True)
    X_oot   = pd.concat(oot_list, ignore_index=True)
    
    # CRITICAL FIX: Re-apply category dtype AFTER concatenation to prevent XGBoost str errors
    for col in CAT_FEATURES:
        X_train[col] = X_train[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_oot[col] = X_oot[col].astype('category')
    
    y_train = X_train.pop('default_flag')
    y_val   = X_val.pop('default_flag')
    y_oot   = X_oot.pop('default_flag')
    
    print(f"Volumes -> Train: {len(X_train):,} | Val: {len(X_val):,} | OOT: {len(X_oot):,}")

    # --- Logistic Regression Prep ---
    X_train_lr = pd.get_dummies(X_train, columns=CAT_FEATURES, drop_first=True)
    X_val_lr   = pd.get_dummies(X_val, columns=CAT_FEATURES, drop_first=True)
    X_oot_lr   = pd.get_dummies(X_oot, columns=CAT_FEATURES, drop_first=True)
    
    X_train_lr, X_val_lr = X_train_lr.align(X_val_lr, join='left', axis=1, fill_value=0)
    X_train_lr, X_oot_lr = X_train_lr.align(X_oot_lr, join='left', axis=1, fill_value=0)
    
    imputer = SimpleImputer(strategy='median')
    X_train_lr = imputer.fit_transform(X_train_lr)
    X_val_lr   = imputer.transform(X_val_lr)
    X_oot_lr   = imputer.transform(X_oot_lr)
    
    scaler = StandardScaler()
    X_train_lr = scaler.fit_transform(X_train_lr)
    X_val_lr   = scaler.transform(X_val_lr)
    X_oot_lr   = scaler.transform(X_oot_lr)

    # --- Train Models ---
    lr_model = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
    lr_model.fit(X_train_lr, y_train)
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        scale_pos_weight=19.4, eval_metric='aucpr',
        enable_categorical=True, n_jobs=-1, random_state=42,
        early_stopping_rounds=30
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    # Save Models
    joblib.dump(lr_model, os.path.join(MODEL_PATH, "lr_challenger.joblib"))
    xgb_model.save_model(os.path.join(MODEL_PATH, "xgb_champion.json"))
    print(f"✅ Models saved to {MODEL_PATH}")

    # --- Evaluation ---
    y_pred_xgb = xgb_model.predict_proba(X_oot)[:, 1]
    auc = roc_auc_score(y_oot, y_pred_xgb)
    pr_auc = average_precision_score(y_oot, y_pred_xgb)
    print(f"\n--- Champion (XGBoost) OOT Backtest ---")
    print(f"Gini      : {2 * auc - 1:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")

train_models()

Training Champion and Challenger models (Memory Safe)...
Volumes -> Train: 4,651,867 | Val: 2,054,045 | OOT: 4,602,682
✅ Models saved to C:\CreditRiskCourse\models\freddie

--- Champion (XGBoost) OOT Backtest ---
Gini      : 0.5070
PR-AUC    : 0.0216


# %% [markdown]
# ## 5. IFRS 9 ECL Engine (Dynamic LGD)
# This engine calculates EAD, assigns IFRS 9 stages, applies a Dynamic LGD based on the HPI macro trend, and provisions the Expected Credit Loss (ECL).

# %%

In [8]:
def run_ifrs9_engine():
    print("\nRunning IFRS 9 ECL Engine (2006-2023)...")
    
    # CRITICAL FIX: Load the champion model from disk to make it available in this scope
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(os.path.join(MODEL_PATH, "xgb_champion.json"))
    
    yearly_results = []
    
    for year in range(2006, 2024):
        file_path = os.path.join(PROCESSED_PATH, f"freddie_master_{year}.parquet")
        if not os.path.exists(file_path): continue
            
        df = pd.read_parquet(file_path, columns=NEEDED_COLS)
        df['GDP_Growth'] = MACRO_DATA[year][0]
        df['Unemployment'] = MACRO_DATA[year][1]
        df['HPI_YoY'] = MACRO_DATA[year][2]
        
        X = df[FEATURES].copy()
        
        # Ensure categorical dtypes for the engine
        for col in CAT_FEATURES: 
            X[col] = X[col].astype('category')
        
        raw_pd = xgb_model.predict_proba(X)[:, 1]
        calibrated_pd = np.clip(raw_pd / 19.4, 0.0001, 0.9999)
        df['PD_12M'] = np.clip(1 - (1 - calibrated_pd)**12, 0, 1)
        
        # Staging
        df['Stage'] = np.select(
            [df['default_flag'] == 1, df['PD_12M'] > 0.15],
            ['Stage 3', 'Stage 2'],
            default='Stage 1'
        )
        
        # Effective PD (Lifetime vs 12M)
        lifetime_pd = np.clip(df['PD_12M'] * 3.0, 0, 1.0)
        df['Effective_PD'] = np.where(df['Stage'] == 'Stage 1', df['PD_12M'], lifetime_pd)
        
        # Dynamic LGD
        hpi = MACRO_DATA[year][2]
        lgd = 0.30 if hpi < -5.0 else (0.25 if hpi < 0 else 0.20)
        
        # ECL
        df['ECL'] = df['Effective_PD'] * lgd * df['CURRENT ACTUAL UPB']
        
        agg = df.groupby('Stage').agg(Balance=('CURRENT ACTUAL UPB', 'sum'), ECL=('ECL', 'sum'))
        
        row = {'Year': year}
        for stage in ['Stage 1', 'Stage 2', 'Stage 3']:
            row[f'{stage}_ECL'] = agg.loc[stage, 'ECL'] if stage in agg.index else 0
            row[f'{stage}_Bal_%'] = (agg.loc[stage, 'Balance'] / agg['Balance'].sum()) * 100 if stage in agg.index else 0
            
        row['Total_ECL'] = agg['ECL'].sum()
        row['Coverage_%'] = (row['Total_ECL'] / agg['Balance'].sum()) * 100
        yearly_results.append(row)
        
        del df, X; gc.collect()

    res_df = pd.DataFrame(yearly_results).set_index('Year')
    
    # Print Summary
    print("\nIFRS 9 STAGING MIGRATION (Balance %):")
    print(res_df[['Stage 1_Bal_%', 'Stage 2_Bal_%', 'Stage 3_Bal_%']].round(2).to_string())
    
    # Plot & Save ECL Chart
    fig, ax1 = plt.subplots(figsize=(12, 6))
    (res_df[['Stage 1_ECL', 'Stage 2_ECL', 'Stage 3_ECL']] / 1e6).plot(
        kind='bar', stacked=True, ax=ax1, color=['#4C72B0', '#DD8452', '#C44E52']
    )
    ax1.set_title('IFRS 9 ECL Provision by Stage (2006-2023)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('ECL Provision (Millions $)')
    
    ax2 = ax1.twinx()
    ax2.plot(range(len(res_df)), res_df['Coverage_%'], color='black', marker='o', linewidth=2, label='Coverage %')
    ax2.set_ylabel('Coverage Ratio (%)')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    chart_path = os.path.join(FIGURES_PATH, "ifrs9_ecl_provision_trend.png")
    plt.savefig(chart_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ ECL Chart saved to: {chart_path}")

run_ifrs9_engine()


Running IFRS 9 ECL Engine (2006-2023)...

IFRS 9 STAGING MIGRATION (Balance %):
      Stage 1_Bal_%  Stage 2_Bal_%  Stage 3_Bal_%
Year                                             
2006          30.85          63.84           5.31
2007          31.73          61.76           6.51
2008          46.36          49.45           4.19
2009          90.87           8.35           0.77
2010          90.70           8.63           0.67
2011          92.96           6.46           0.58
2012          96.94           2.62           0.44
2013          96.09           3.43           0.48
2014          95.15           4.19           0.65
2015          92.00           7.34           0.66
2016          92.68           6.57           0.76
2017          89.42           9.30           1.28
2018          83.86          14.50           1.64
2019          90.33           8.03           1.64
2020          99.41           0.19           0.40
2021          98.34           1.35           0.31
2022          73.57